# NB17: Cross-Dataset Synthesis

Compiles results from NB01–NB13 into a unified cross-dataset analysis.
The only new computations are:
1. Pooled THINGS + LITTLE THINGS Wilcoxon test (N=22)
2. Baryonic mass demographics comparison
3. Formal outcome scenario determination

All other numbers are loaded from upstream CSVs.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

from src.btfr import (
    compute_btfr_residuals, run_btfr_covariance_test,
    compute_mbar_for_sample, BTFR_ALPHA, BTFR_BETA,
)
from src.database import get_session, query_fits_as_dataframe
from src.utils import get_project_root

root = get_project_root()
results_dir = root / "results"

# Constants
A0 = 1.2e-10        # m/s^2
A0_HALF = A0 / 2     # 6.00e-11 m/s^2

print(f"Project root: {root}")
print(f"Results dir:  {results_dir}")

Project root: C:\Science\RT\a02-transition-scale
Results dir:  C:\Science\RT\a02-transition-scale\results


## 1. Load Upstream Results

In [2]:
# --- Per-galaxy g(Rt) results ---
sparc_df = pd.read_csv(results_dir / "NB01_baseline_results.csv")
things_df = pd.read_csv(results_dir / "NB09_things_per_galaxy.csv")
lt_df = pd.read_csv(results_dir / "NB13_lt_per_galaxy.csv")

print(f"SPARC resolved:        {len(sparc_df)} galaxies")
print(f"THINGS valid g(Rt):    {len(things_df)} galaxies")
print(f"LITTLE THINGS valid:   {len(lt_df)} galaxies")

# --- BTFR summary results ---
things_btfr = pd.read_csv(results_dir / "NB09_things_btfr_results.csv")
lt_btfr = pd.read_csv(results_dir / "NB13_lt_btfr_results.csv")

# --- BIC comparison files ---
things_bic = pd.read_csv(results_dir / "NB10_things_bic_comparison.csv")
lt_bic = pd.read_csv(results_dir / "NB12_lt_bic_comparison.csv")

# --- Cross-pipeline validation ---
cross_df = pd.read_csv(results_dir / "NB08_cross_pipeline.csv")
both_resolved_df = pd.read_csv(results_dir / "NB08_both_resolved.csv")

print(f"\nCross-pipeline overlap: {len(cross_df)} galaxies")
print(f"Both resolved:         {len(both_resolved_df)} galaxies")

SPARC resolved:        98 galaxies
THINGS valid g(Rt):    8 galaxies
LITTLE THINGS valid:   14 galaxies

Cross-pipeline overlap: 13 galaxies
Both resolved:         5 galaxies


In [3]:
# Compute SPARC BTFR residuals (NB02 didn't export per-galaxy CSV)
sparc_mbar = compute_mbar_for_sample(sparc_df["galaxy_id"].tolist())
sparc_g = sparc_df.set_index("galaxy_id")["g_Rt_mks"]

# Align: only galaxies present in both
sparc_ids = [gid for gid in sparc_df["galaxy_id"] if gid in sparc_mbar]
sparc_g_arr = np.array([sparc_g[gid] for gid in sparc_ids])
sparc_m_arr = np.array([sparc_mbar[gid] for gid in sparc_ids])

sparc_residuals = compute_btfr_residuals(sparc_g_arr, sparc_m_arr)
sparc_btfr_result = run_btfr_covariance_test(sparc_g_arr, sparc_m_arr)

print(f"SPARC BTFR test (N={sparc_btfr_result['n_galaxies']}):\n")
print(f"  Median residual:  {sparc_btfr_result['median_residual']:.4f} dex")
print(f"  Raw scatter:      {sparc_btfr_result['scatter_raw']:.3f} dex")
print(f"  Residual scatter: {sparc_btfr_result['scatter_residual']:.3f} dex")
print(f"  Wilcoxon p:       {sparc_btfr_result['wilcoxon_pvalue']:.3f}")

2026-04-09 23:56:25 | INFO     | src.ingest | Loaded 175 galaxies from SPARC metadata


2026-04-09 23:56:25 | INFO     | src.ingest | Loaded 175 bulge luminosities


SPARC BTFR test (N=98):

  Median residual:  -0.0040 dex
  Raw scatter:      0.409 dex
  Residual scatter: 0.277 dex
  Wilcoxon p:       0.654


## 2. Cross-Dataset Summary Table

In [4]:
sparc_median_g = np.median(sparc_g_arr)
sparc_offset = (sparc_median_g - A0_HALF) / A0_HALF * 100

things_row = things_btfr.iloc[0]
lt_row = lt_btfr.iloc[0]

summary = pd.DataFrame([
    {
        "Dataset": "SPARC",
        "N_total": 175,
        "N_fitted": 175,
        "N_resolved": 98,
        "N_valid_g": sparc_btfr_result["n_galaxies"],
        "Median_g_mks": sparc_median_g,
        "Offset_from_a0half_pct": sparc_offset,
        "Scatter_raw_dex": sparc_btfr_result["scatter_raw"],
        "Scatter_residual_dex": sparc_btfr_result["scatter_residual"],
        "Median_residual_dex": sparc_btfr_result["median_residual"],
        "Wilcoxon_p": sparc_btfr_result["wilcoxon_pvalue"],
        "Outcome": "BASELINE (anchored)",
    },
    {
        "Dataset": "THINGS",
        "N_total": 19,
        "N_fitted": 17,
        "N_resolved": 10,
        "N_valid_g": int(things_row["n_galaxies"]),
        "Median_g_mks": things_row["median_g_mks"],
        "Offset_from_a0half_pct": things_row["offset_pct"],
        "Scatter_raw_dex": things_row["scatter_raw_dex"],
        "Scatter_residual_dex": things_row["scatter_residual_dex"],
        "Median_residual_dex": things_row["median_residual_dex"],
        "Wilcoxon_p": things_row["wilcoxon_pvalue"],
        "Outcome": things_row["outcome"],
    },
    {
        "Dataset": "LITTLE THINGS",
        "N_total": 26,
        "N_fitted": 20,
        "N_resolved": 15,
        "N_valid_g": int(lt_row["n_galaxies"]),
        "Median_g_mks": lt_row["median_g_mks"],
        "Offset_from_a0half_pct": lt_row["offset_pct"],
        "Scatter_raw_dex": lt_row["scatter_raw_dex"],
        "Scatter_residual_dex": lt_row["scatter_residual_dex"],
        "Median_residual_dex": lt_row["median_residual_dex"],
        "Wilcoxon_p": lt_row["wilcoxon_pvalue"],
        "Outcome": lt_row["outcome"],
    },
])

summary.to_csv(results_dir / "NB17_cross_dataset_summary.csv", index=False)
print("Saved: NB17_cross_dataset_summary.csv\n")
summary

Saved: NB17_cross_dataset_summary.csv



,Dataset,N_total,N_fitted,N_resolved,N_valid_g,Median_g_mks,Offset_from_a0half_pct,Scatter_raw_dex,Scatter_residual_dex,Median_residual_dex,Wilcoxon_p,Outcome
0,SPARC,175,175,98,98,6.512157e-11,8.535945,0.408911,0.277452,-0.003984,0.653964,BASELINE (anchored)
1,THINGS,19,17,10,8,8.734848e-11,45.574517,0.345517,0.210477,0.049291,0.546875,INCONCLUSIVE
2,LITTLE THINGS,26,20,15,14,1.656786e-11,-72.388098,0.280099,0.245250,-0.241669,0.049438,REJECTED


## 3. Pooled THINGS + LITTLE THINGS Test (N=22)

The pre-registration required the confirmatory test to use independent data.
With THINGS underpowered (N=8) and LITTLE THINGS rejecting (N=14), pooling
gives the maximum-power combined test against the pre-registered threshold.

In [5]:
things_residuals = things_df["residual"].values
lt_residuals = lt_df["residual"].values

pooled_residuals = np.concatenate([things_residuals, lt_residuals])
pooled_n = len(pooled_residuals)
pooled_median = np.median(pooled_residuals)

# Wilcoxon signed-rank test on pooled residuals
nonzero = pooled_residuals[pooled_residuals != 0]
pooled_stat, pooled_p = stats.wilcoxon(nonzero, alternative="two-sided")

# Pre-registered threshold: +/-0.041 dex (corresponds to +/-10%)
THRESHOLD_DEX = 0.041
passes_threshold = abs(pooled_median) <= THRESHOLD_DEX

print(f"Pooled test (N={pooled_n}):")
print(f"  THINGS contribution:        N={len(things_residuals)}")
print(f"  LITTLE THINGS contribution: N={len(lt_residuals)}")
print(f"  Median residual:            {pooled_median:.4f} dex")
print(f"  Wilcoxon statistic:         {pooled_stat:.1f}")
print(f"  Wilcoxon p-value:           {pooled_p:.4f}")
print(f"  Within +/-{THRESHOLD_DEX} dex threshold: {passes_threshold}")
print(f"  Outcome: {'PASS' if passes_threshold and pooled_p > 0.05 else 'FAIL'}")

# Percentile range
p16, p84 = np.percentile(pooled_residuals, [16, 84])
pooled_scatter = (p84 - p16) / 2.0
print(f"  Scatter (half p16-p84):     {pooled_scatter:.3f} dex")

# Build per-galaxy pooled table
pooled_galaxies = pd.DataFrame({
    "galaxy_id": list(things_df["galaxy_id"]) + list(lt_df["galaxy_id"]),
    "dataset": ["THINGS"] * len(things_df) + ["LITTLE_THINGS"] * len(lt_df),
    "g_Rt_mks": list(things_df["g_Rt_mks"]) + list(lt_df["g_Rt_mks"]),
    "M_bar": list(things_df["M_bar"]) + list(lt_df["M_bar"]),
    "residual": list(pooled_residuals),
})

pooled_galaxies.to_csv(results_dir / "NB17_pooled_test.csv", index=False)
print(f"\nSaved: NB17_pooled_test.csv ({len(pooled_galaxies)} galaxies)")

Pooled test (N=22):
  THINGS contribution:        N=8
  LITTLE THINGS contribution: N=14
  Median residual:            -0.0818 dex
  Wilcoxon statistic:         83.0
  Wilcoxon p-value:           0.1659
  Within +/-0.041 dex threshold: False
  Outcome: FAIL
  Scatter (half p16-p84):     0.253 dex

Saved: NB17_pooled_test.csv (22 galaxies)


## 4. Baryonic Mass Demographics

The key to understanding the null result: different datasets have different
baryonic mass distributions, and g(Rt) scales with mass.

In [6]:
sparc_log_mbar = np.log10(sparc_m_arr)
things_log_mbar = np.log10(things_df["M_bar"].values)
lt_log_mbar = np.log10(lt_df["M_bar"].values)

demographics = pd.DataFrame([
    {
        "Dataset": "SPARC",
        "N_valid_g": len(sparc_m_arr),
        "Median_log_Mbar": np.median(sparc_log_mbar),
        "Median_Mbar_Msun": 10**np.median(sparc_log_mbar),
        "Min_log_Mbar": np.min(sparc_log_mbar),
        "Max_log_Mbar": np.max(sparc_log_mbar),
        "Median_g_Rt_mks": np.median(sparc_g_arr),
    },
    {
        "Dataset": "THINGS",
        "N_valid_g": len(things_df),
        "Median_log_Mbar": np.median(things_log_mbar),
        "Median_Mbar_Msun": 10**np.median(things_log_mbar),
        "Min_log_Mbar": np.min(things_log_mbar),
        "Max_log_Mbar": np.max(things_log_mbar),
        "Median_g_Rt_mks": np.median(things_df["g_Rt_mks"].values),
    },
    {
        "Dataset": "LITTLE THINGS",
        "N_valid_g": len(lt_df),
        "Median_log_Mbar": np.median(lt_log_mbar),
        "Median_Mbar_Msun": 10**np.median(lt_log_mbar),
        "Min_log_Mbar": np.min(lt_log_mbar),
        "Max_log_Mbar": np.max(lt_log_mbar),
        "Median_g_Rt_mks": np.median(lt_df["g_Rt_mks"].values),
    },
])

demographics.to_csv(results_dir / "NB17_mass_demographics.csv", index=False)
print("Saved: NB17_mass_demographics.csv\n")
demographics

Saved: NB17_mass_demographics.csv



,Dataset,N_valid_g,Median_log_Mbar,Median_Mbar_Msun,Min_log_Mbar,Max_log_Mbar,Median_g_Rt_mks
0,SPARC,98,9.763214,5.797149e+09,7.690728,11.422913,6.512157e-11
1,THINGS,8,10.088855,1.227029e+10,8.593563,11.065481,8.734848e-11
2,LITTLE THINGS,14,8.336338,2.169394e+08,6.391288,9.178687,1.656786e-11


In [7]:
# What M_bar would a sample need to have median g(Rt) = a0/2?
# From the BTFR: log10(a0/2) = alpha * log10(M_bar) + beta
log_a0half = np.log10(A0_HALF)
log_mbar_target = (log_a0half - BTFR_BETA) / BTFR_ALPHA
mbar_target = 10**log_mbar_target

print(f"BTFR trend crosses a0/2 at:")
print(f"  log10(M_bar) = ({log_a0half:.4f} - ({BTFR_BETA})) / {BTFR_ALPHA} = {log_mbar_target:.2f}")
print(f"  M_bar = {mbar_target:.2e} Msun")
print(f"\nSPARC median log10(M_bar) = {np.median(sparc_log_mbar):.2f}")
print(f"  => SPARC sits right at the crossing point \u2014 the alignment is demographic")

BTFR trend crosses a0/2 at:
  log10(M_bar) = (-10.2218 - (-12.55)) / 0.238 = 9.78
  M_bar = 6.06e+09 Msun

SPARC median log10(M_bar) = 9.76
  => SPARC sits right at the crossing point — the alignment is demographic


## 5. Constrained Model Cross-Dataset Summary

In [8]:
# SPARC constrained model stats from database
session = get_session()
sparc_con_fits = query_fits_as_dataframe(session, model_name="constrained_rt")
sparc_con_fits = sparc_con_fits[
    ~sparc_con_fits["galaxy_id"].str.contains("THINGS", na=False)
]
sparc_free_fits = query_fits_as_dataframe(session, model_name="rational_taper")
sparc_free_fits = sparc_free_fits[
    ~sparc_free_fits["galaxy_id"].str.contains("THINGS", na=False)
]
session.close()

# SPARC no-solution rate
sparc_con_total = len(sparc_con_fits)
sparc_con_converged = int(sparc_con_fits["converged"].sum())
sparc_no_solution_rate = (sparc_con_total - sparc_con_converged) / sparc_con_total * 100

# SPARC delta_BIC (only where both converged)
sparc_con_ok = sparc_con_fits[sparc_con_fits["converged"]][["galaxy_id", "bic"]]
sparc_bic_merged = sparc_free_fits[["galaxy_id", "bic"]].merge(
    sparc_con_ok, on="galaxy_id", suffixes=("_free", "_con")
)
sparc_bic_merged["delta_bic"] = sparc_bic_merged["bic_con"] - sparc_bic_merged["bic_free"]
sparc_median_dbic = sparc_bic_merged["delta_bic"].median()

# THINGS constrained model stats — load constrained fits CSV (has both converged/non-converged)
things_con_all = pd.read_csv(results_dir / "NB10_things_constrained_fits.csv")
things_con_total = len(things_con_all)
things_con_converged = int(things_con_all["converged"].sum())
things_no_solution_rate = (things_con_total - things_con_converged) / things_con_total * 100
things_median_dbic = things_bic["delta_bic"].median()  # BIC CSV has only converged pairs

# LITTLE THINGS constrained model stats (all 20 converged per NB12)
lt_con_total = 20
lt_con_converged = 20
lt_no_solution_rate = 0.0
lt_median_dbic = lt_bic["delta_bic"].median()

constrained_summary = pd.DataFrame([
    {
        "Dataset": "SPARC",
        "N_total": sparc_con_total,
        "N_converged": sparc_con_converged,
        "No_solution_rate_pct": sparc_no_solution_rate,
        "Median_delta_BIC": sparc_median_dbic,
        "N_free_wins": int((sparc_bic_merged["delta_bic"] > 0).sum()),
        "N_constrained_wins": int((sparc_bic_merged["delta_bic"] < 0).sum()),
    },
    {
        "Dataset": "THINGS",
        "N_total": things_con_total,
        "N_converged": things_con_converged,
        "No_solution_rate_pct": things_no_solution_rate,
        "Median_delta_BIC": things_median_dbic,
        "N_free_wins": int((things_bic["delta_bic"] > 0).sum()),
        "N_constrained_wins": int((things_bic["delta_bic"] < 0).sum()),
    },
    {
        "Dataset": "LITTLE THINGS",
        "N_total": lt_con_total,
        "N_converged": lt_con_converged,
        "No_solution_rate_pct": lt_no_solution_rate,
        "Median_delta_BIC": lt_median_dbic,
        "N_free_wins": int((lt_bic["delta_bic"] > 0).sum()),
        "N_constrained_wins": int((lt_bic["delta_bic"] < 0).sum()),
    },
])

constrained_summary.to_csv(results_dir / "NB17_constrained_summary.csv", index=False)
print("Saved: NB17_constrained_summary.csv\n")
constrained_summary

Saved: NB17_constrained_summary.csv


,Dataset,N_total,N_converged,No_solution_rate_pct,Median_delta_BIC,N_free_wins,N_constrained_wins
0,SPARC,123,31,74.796748,96.880187,30,1
1,THINGS,17,4,76.470588,456.858306,3,1
2,LITTLE THINGS,20,20,0.000000,309.295636,19,1


## 6. Outcome Scenario Determination

The pre-registration defined three outcome scenarios:
- **Scenario A (Strong replication):** Median residual within ±10% in independent data, Wilcoxon does not reject
- **Scenario B (Partial replication):** Alignment weakens but does not clearly fail
- **Scenario C (BTFR artifact):** Alignment fails in independent data; g(Rt) is mass-dependent, not universal

In [9]:
print("=" * 70)
print("OUTCOME SCENARIO EVALUATION")
print("=" * 70)

print("\n--- Individual dataset tests ---")
print(f"THINGS (NB09):  median residual = {things_row['median_residual_dex']:+.3f} dex, "
      f"p = {things_row['wilcoxon_pvalue']:.3f}, outcome = {things_row['outcome']}")
print(f"LITTLE THINGS (NB13): median residual = {lt_row['median_residual_dex']:+.3f} dex, "
      f"p = {lt_row['wilcoxon_pvalue']:.3f}, outcome = {lt_row['outcome']}")

print(f"\n--- Pooled test (N={pooled_n}) ---")
print(f"Median residual = {pooled_median:+.4f} dex")
print(f"Wilcoxon p = {pooled_p:.4f}")
print(f"Within +/-{THRESHOLD_DEX} dex: {passes_threshold}")

print(f"\n--- Constrained model ---")
print(f"Decisively rejected in all three datasets (median \u0394BIC > +96 everywhere)")

# Formal determination
scenario_a = passes_threshold and pooled_p > 0.05
scenario_c = (not passes_threshold) or (lt_row["outcome"] == "REJECTED")
scenario_b = not scenario_a and not scenario_c

if scenario_a:
    outcome = "A"
    description = "Strong replication: a0/2 alignment holds across datasets"
elif scenario_c:
    outcome = "C"
    description = "BTFR artifact: a0/2 alignment is mass-dependent, driven by SPARC demographics"
else:
    outcome = "B"
    description = "Partial replication: alignment weakens but does not clearly fail"

print(f"\n{'=' * 70}")
print(f"CONFIRMED OUTCOME: Scenario {outcome}")
print(f"{description}")
print(f"{'=' * 70}")

outcome_df = pd.DataFrame([{
    "Outcome_scenario": outcome,
    "Description": description,
    "Pooled_N": pooled_n,
    "Pooled_median_residual_dex": pooled_median,
    "Pooled_wilcoxon_p": pooled_p,
    "Pooled_passes_threshold": passes_threshold,
    "THINGS_outcome": things_row["outcome"],
    "LT_outcome": lt_row["outcome"],
}])

outcome_df.to_csv(results_dir / "NB17_outcome_scenario.csv", index=False)
print("\nSaved: NB17_outcome_scenario.csv")

OUTCOME SCENARIO EVALUATION

--- Individual dataset tests ---
THINGS (NB09):  median residual = +0.049 dex, p = 0.547, outcome = INCONCLUSIVE
LITTLE THINGS (NB13): median residual = -0.242 dex, p = 0.049, outcome = REJECTED

--- Pooled test (N=22) ---
Median residual = -0.0818 dex
Wilcoxon p = 0.1659
Within +/-0.041 dex: False

--- Constrained model ---
Decisively rejected in all three datasets (median ΔBIC > +96 everywhere)

CONFIRMED OUTCOME: Scenario C
BTFR artifact: a0/2 alignment is mass-dependent, driven by SPARC demographics

Saved: NB17_outcome_scenario.csv


## 7. DDO 154 Triple-Overlap Comparison

DDO 154 appears in all three datasets, serving as a cross-pipeline consistency check.

In [10]:
# DDO 154 in SPARC
sparc_ddo154 = sparc_df[sparc_df["galaxy_id"] == "DDO154"]
sparc_ddo154_g = sparc_ddo154["g_Rt_mks"].values[0] if len(sparc_ddo154) > 0 else np.nan

# DDO 154 in THINGS
things_ddo154 = things_df[things_df["galaxy_id"] == "DDO154_THINGS"]
things_ddo154_g = things_ddo154["g_Rt_mks"].values[0] if len(things_ddo154) > 0 else np.nan

# DDO 154 in LITTLE THINGS
lt_ddo154 = lt_df[lt_df["galaxy_id"] == "DDO_154_LITTLE_THINGS"]
lt_ddo154_g = lt_ddo154["g_Rt_mks"].values[0] if len(lt_ddo154) > 0 else np.nan

print("DDO 154 g(Rt) across three pipelines:")
print(f"  SPARC:         {sparc_ddo154_g:.2e} m/s\u00b2")
print(f"  THINGS:        {things_ddo154_g:.2e} m/s\u00b2")
print(f"  LITTLE THINGS: {lt_ddo154_g:.2e} m/s\u00b2")
print(f"  a0/2:          {A0_HALF:.2e} m/s\u00b2")
print(f"\nAll three are well below a0/2 \u2014 consistent with DDO 154 being a low-mass dwarf.")

# Log-space spread
ddo154_vals = np.array([sparc_ddo154_g, things_ddo154_g, lt_ddo154_g])
ddo154_vals = ddo154_vals[np.isfinite(ddo154_vals)]
if len(ddo154_vals) >= 2:
    spread = np.max(np.log10(ddo154_vals)) - np.min(np.log10(ddo154_vals))
    print(f"  Log-space spread: {spread:.3f} dex")

DDO 154 g(Rt) across three pipelines:
  SPARC:         1.87e-11 m/s²
  THINGS:        1.48e-11 m/s²
  LITTLE THINGS: 2.12e-11 m/s²
  a0/2:          6.00e-11 m/s²

All three are well below a0/2 — consistent with DDO 154 being a low-mass dwarf.
  Log-space spread: 0.155 dex


## Summary

**Outcome: Scenario C (BTFR artifact)**

The $a_0/2$ alignment discovered in Paper 2 does not replicate in independent datasets.
The transition acceleration $g(R_t)$ is mass-dependent, following the BTFR scaling with
slope 0.238. SPARC's alignment with $a_0/2$ is a demographic coincidence: its median
baryonic mass places galaxies exactly where the BTFR trend crosses $a_0/2$.

The RT model itself remains a competitive rotation curve fitting tool across all three
datasets. Only the specific claim that $R_t$ occurs at a universal acceleration scale
is retracted.

### Output files
- `NB17_cross_dataset_summary.csv` — Master comparison table
- `NB17_pooled_test.csv` — Per-galaxy pooled THINGS + LITTLE THINGS residuals
- `NB17_mass_demographics.csv` — Baryonic mass demographics per dataset
- `NB17_constrained_summary.csv` — Constrained model BIC comparison
- `NB17_outcome_scenario.csv` — Formal outcome determination